# D-Wave Quantum Annealing for MTDA

Solve the MTDA QUBO problem using D-Wave quantum annealing:
- Neal simulator (local)
- D-Wave Advantage2 (cloud, requires token)
- Reverse annealing warm-start
- Comparison with classical Hungarian algorithm

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from quantum_common.visualization.styles import apply_publication_style, QUANTUM_COLORS

apply_publication_style()

from quantum_mht.formulation.mtda_qubo_builder import MTDAQuboBuilder
from quantum_mht.solvers.solver_factory import create_solver

In [ ]:
# Create a 5-target problem
rng = np.random.default_rng(42)
n_targets = 5
predicted = rng.uniform(10, 90, size=(n_targets, 2))
measurements = np.vstack([
    predicted + rng.normal(0, 2, size=(n_targets, 2)),
    rng.uniform(0, 100, size=(3, 2)),  # 3 clutter points
])
covs = np.array([np.eye(2) * 4.0 for _ in range(n_targets)])

builder = MTDAQuboBuilder()
cost_matrix = builder.cost_builder.build(predicted, measurements, covs)
qubo = builder.build_from_cost_matrix(cost_matrix)
print(f"QUBO: {qubo.num_variables} variables")

In [ ]:
# Solve with D-Wave neal simulator
try:
    annealing = create_solver('annealing', use_simulator=True, num_reads=500)
    result_anneal = annealing.solve(qubo)
    print(f"Annealing: obj={result_anneal.objective_value:.2f}, time={result_anneal.solve_time_s:.4f}s")
    print(f"  Assignments: {result_anneal.assignments}")
    print(f"  Missed: {result_anneal.missed_detections}")
    print(f"  False alarms: {result_anneal.false_alarms}")
    print(f"  Feasible: {result_anneal.is_feasible}")
except Exception as e:
    print(f"Annealing failed: {e}")

In [ ]:
# Compare with Hungarian (optimal classical)
hungarian = create_solver('hungarian')
result_hung = hungarian.solve(qubo)
print(f"Hungarian: obj={result_hung.objective_value:.2f}, time={result_hung.solve_time_s:.4f}s")
print(f"  Assignments: {result_hung.assignments}")

# Compare
if 'result_anneal' in dir():
    gap = abs(result_anneal.objective_value - result_hung.objective_value)
    print(f"\nEnergy gap (annealing - hungarian): {gap:.4f}")
    speedup = result_hung.solve_time_s / max(result_anneal.solve_time_s, 1e-6)
    print(f"Speed ratio: {speedup:.2f}x")